# KPSS modeli — LoRA ince ayarı

Bu defter, açık ağırlıklı bir modeli KPSS çalışma notu ve ÖSYM tarzı soru üretmek üzere eğitir.
Sonuçta senin Hugging Face hesabında duran, uygulamanın doğrudan kullanabileceği bir model çıkar.

**Nerede çalıştırılır:** Google Colab, ücretsiz T4 GPU.
`Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU` seçin.

**Önce veri seti hazırlanır** (kendi bilgisayarında):

```powershell
# backend/.env icinde: CAPTURE_TRAINING_DATA=true ve guclu bir saglayici (gemini/huggingface)
# birkac ders videosunu analiz ettikten sonra:
cd training
python build_dataset.py
```

Oluşan `train.jsonl` ve `val.jsonl` dosyalarını bu deftere yükleyeceksin.

**Eğitim mantığı:** Model, uygulamanın çalışma anında kullandığı istemin birebir aynısını görür
ve beklenen JSON çıktısını üretmeyi öğrenir. Böylece küçük model, kendisinden çok daha büyük bir
modelin biçim disiplinini ve üslubunu taklit eder.

## 1. Kurulum

Unsloth, LoRA eğitimini T4 gibi küçük kartlarda mümkün kılar: modeli 4 bit'e sıkıştırır ve
bellek kullanımını yaklaşık dörtte bire indirir.

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps --upgrade "trl>=0.15.0" peft accelerate bitsandbytes datasets

import torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## 2. Veri setini yükle

Soldaki dosya simgesinden `train.jsonl` ve `val.jsonl` dosyalarını sürükleyip bırak.

In [ ]:
import json
import os

from datasets import load_dataset

assert os.path.exists("train.jsonl"), "train.jsonl yuklenmedi"

data_files = {"train": "train.jsonl"}
if os.path.exists("val.jsonl") and os.path.getsize("val.jsonl") > 0:
    data_files["validation"] = "val.jsonl"

dataset = load_dataset("json", data_files=data_files)
print(dataset)

ornek = dataset["train"][0]
print("\ngorev:", ornek["task"])
print("istem (ilk 300):", ornek["messages"][1]["content"][:300])
print("yanit (ilk 300):", ornek["messages"][2]["content"][:300])

## 3. Temel modeli yükle

`Qwen2.5-7B-Instruct` Türkçeyi iyi bilir, JSON disiplini güçlüdür ve 4 bit hâlinde T4'e sığar.
Daha iyi Türkçe için `unsloth/gemma-2-9b-it-bnb-4bit`, daha hızlı deneme için
`unsloth/Qwen2.5-3B-Instruct-bnb-4bit` kullanılabilir.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 8192
BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

## 4. LoRA katmanlarını ekle

Modelin 7 milyar ağırlığının tamamını eğitmek yerine, aralara küçük eğitilebilir katmanlar
yerleştiriyoruz. Böylece güncellenen parametre sayısı %1'in altına iner; T4'te saatler yerine
dakikalar sürer ve sonuç dosyası birkaç yüz megabayt olur.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

egitilebilir = sum(p.numel() for p in model.parameters() if p.requires_grad)
toplam = sum(p.numel() for p in model.parameters())
print(f"egitilebilir: {egitilebilir:,} / {toplam:,}  (%{100 * egitilebilir / toplam:.2f})")

## 5. Sohbet biçimini uygula

Her örnek, modelin kendi sohbet şablonuna çevrilir. Uzunluk sınırını aşan örnekler elenir;
yarıda kesilmiş bir JSON, modele bozuk çıktı üretmeyi öğretir.

In [ ]:
def bicimle(batch):
    metinler = [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in batch["messages"]
    ]
    return {"text": metinler}


dataset = dataset.map(bicimle, batched=True, remove_columns=["messages", "task"])


def sigiyor(ornek):
    return len(tokenizer(ornek["text"])["input_ids"]) <= MAX_SEQ_LEN


onceki = len(dataset["train"])
dataset = dataset.filter(sigiyor)
print(f"uzunluk elemesi: {onceki} -> {len(dataset['train'])}")
print(dataset["train"][0]["text"][:400])

## 6. Eğitim

`train_on_responses_only`, kaybın yalnızca asistan yanıtı üzerinden hesaplanmasını sağlar.
Bu olmazsa model uzun istemi ezberlemeye çalışır ve asıl öğrenmesi gereken JSON çıktısına
daha az odaklanır.

`num_train_epochs` değerini veri miktarına göre ayarla: birkaç yüz örnekte 3, binlerce
örnekte 1-2 tur yeterlidir.

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

config = SFTConfig(
    output_dir="ciktilar",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_ratio=0.05,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset.get("validation"),
    args=config,
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

istatistik = trainer.train()
print(istatistik)

## 7. Sınama: model geçerli JSON üretiyor mu?

İnce ayarın asıl ölçütü budur. Model doğru alanları içeren, ayrıştırılabilir JSON
üretmiyorsa uygulama onu kullanamaz.

In [ ]:
FastLanguageModel.for_inference(model)

SISTEM = (
    "Sen 20 yillik KPSS egitmeni ve hafiza teknikleri uzmanisin. "
    "Cikti SADECE gecerli JSON olmali."
)
ISTEM = """Konu / ders: Anayasa

Zaman damgali altyazi (her satir: [saniye] metin):
---
[0] Yasama yetkisi Turkiye Buyuk Millet Meclisine aittir ve devredilemez.
[18] Yurutme yetkisi ve gorevi Cumhurbaskani tarafindan kullanilir.
---

Cikti JSON semasi:
{"notes": [{"title": "...", "detail": "...", "key_points": ["..."], "mnemonic": "...", "exam_tip": "...", "timestamp": 0}]}
"""

girdi = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": SISTEM},
        {"role": "user", "content": ISTEM},
    ],
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

cikti = model.generate(input_ids=girdi, max_new_tokens=1200, temperature=0.4, do_sample=True)
yanit = tokenizer.decode(cikti[0][girdi.shape[1] :], skip_special_tokens=True)
print(yanit[:1500])

try:
    veri = json.loads(yanit)
    print("\nGECERLI JSON — not sayisi:", len(veri.get("notes", [])))
except json.JSONDecodeError as hata:
    print("\nGECERSIZ JSON:", hata)

## 8. Hugging Face hesabına yükle

Token'ı [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) adresinden
**write** yetkisiyle al.

In [ ]:
from huggingface_hub import login

HF_TOKEN = "hf_..."  # write yetkili token
REPO = "kullanici-adin/kpss-qwen-7b"

login(HF_TOKEN)

# Sadece LoRA katmanlari (kucuk, hizli)
model.push_to_hub(REPO, token=HF_TOKEN)
tokenizer.push_to_hub(REPO, token=HF_TOKEN)
print("yuklendi:", REPO)

## 9. Kendi bilgisayarında çalıştırmak için GGUF'a çevir

Bu adım, eğitilmiş modeli Ollama'nın okuyabildiği tek dosyaya dönüştürür. İndirdikten sonra
uygulaman internetsiz ve kotasız çalışır.

In [ ]:
model.save_pretrained_gguf("kpss-model", tokenizer, quantization_method="q4_k_m")

modelfile = """FROM ./kpss-model/unsloth.Q4_K_M.gguf
PARAMETER temperature 0.4
PARAMETER num_ctx 8192
"""
with open("kpss-model/Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile)

print("Indirdikten sonra kendi bilgisayarinda:")
print("  ollama create kpss -f Modelfile")
print("  backend/.env icinde: LLM_PROVIDER=ollama ve OLLAMA_MODEL=kpss")

## Sonraki adımlar

1. Eğitilen modeli uygulamada dene: `OLLAMA_MODEL=kpss`
2. Zayıf kaldığı konularda daha çok veri topla ve eğitimi tekrarla.
3. Kaliteyi ölçmek için sabit bir sınama seti tut: aynı 20 altyazı parçasını her sürümde
   çalıştır, geçerli JSON oranını ve soru kalitesini karşılaştır.

Veri miktarı için kaba ölçüt: 200 örnek biçimi öğretir, 1000+ örnek üslubu ve konu
kapsamını gerçekten oturtur.